## Intertemporal decomposition notebook
- Step two of the decomposition pipeline
- Run this after you've ran the parse_experiment_in_individual_files.py
- After this you can run CB

In [1]:
import os
import pandas as pd
import numpy as np
from utils import decomposition

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
td = decomposition.TemporalDecomposition()

In [4]:
# Set up paths
SCRIPT_DIR_PATH = os.getcwd()
CW_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "cw")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
ENSEMBLE_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "ensemble_data")

In [5]:
# Load emissions targets
te_all = pd.read_csv(os.path.join(CW_DIR_PATH, "emission_targets_louisiana.csv"))
target_country = "LA"
cols_needed = ["Subsector", "Gas", "Vars", "Edgar_Class", target_country]
te_all = te_all[cols_needed].copy()
te_all["tvalue"] = te_all[target_country]
te_all = te_all.drop(columns=[target_country])
te_all

,Subsector,Gas,Vars,Edgar_Class,tvalue
0,lvst,ch4,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock:CH4,1.539811
1,lsmm,ch4,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock:CH4,0.151107
2,lsmm,n2o,emission_co2e_n2o_lsmm_direct_anaerobic_digest...,AG - Livestock:N2O,0.079212
3,agrc,co2,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops:CO2,0.000000
4,agrc,ch4,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops:CH4,2.384974
...,...,...,...,...,...
63,soil,co2,emission_co2e_co2_soil_lime_use:emission_co2e_...,LULUCF - Organic Soil:CO2,0.305152
64,soil,n2o,emission_co2e_n2o_soil_fertilizer:emission_co2...,LULUCF - Organic Soil:N2O,0.928029
65,ccsq,ch4,emission_co2e_ch4_ccsq_direct_air_capture,CCSQ:CH4,0.000000
66,ccsq,co2,emission_co2e_co2_ccsq_direct_air_capture,CCSQ:CO2,0.000000


In [6]:
# Parse target variables
te_all["Vars_list"] = te_all["Vars"].str.split(":")
target_vars = [item for sublist in te_all["Vars_list"].tolist() for item in sublist]
print("Target variables:", target_vars[:10])  # Display first 10 target variables
print("Total target variables:", len(target_vars))

Target variables: ['emission_co2e_ch4_lvst_entferm_buffalo', 'emission_co2e_ch4_lvst_entferm_cattle_dairy', 'emission_co2e_ch4_lvst_entferm_cattle_nondairy', 'emission_co2e_ch4_lvst_entferm_chickens', 'emission_co2e_ch4_lvst_entferm_goats', 'emission_co2e_ch4_lvst_entferm_horses', 'emission_co2e_ch4_lvst_entferm_mules', 'emission_co2e_ch4_lvst_entferm_pigs', 'emission_co2e_ch4_lvst_entferm_sheep', 'emission_co2e_ch4_lsmm_anaerobic_digester']
Total target variables: 469


In [8]:
# Output folders
ensemble_id = "2025-08-28t15;29;22.344855" #NOTE: Change this to your ensemble ID
RUN_ENSEMBLE_DIR_PATH = os.path.join(ENSEMBLE_DATA_DIR_PATH, f"sisepuede_summary_results_run_sisepuede_run_{ensemble_id}")
PARSED_RUNS_DIR_PATH = os.path.join(DATA_DIR_PATH, "parsed_runs")
ENSEMBLE_PARSED_DIR_PATH = os.path.join(PARSED_RUNS_DIR_PATH, ensemble_id)
files_names = [f for f in os.listdir(ENSEMBLE_PARSED_DIR_PATH) if f.endswith('.csv')]

In [9]:
print(f"Number of parsed run files found: {len(files_names)}")

Number of parsed run files found: 3004


In [10]:
PARSED_RUNS_PROCESSED_DIR_PATH = os.path.join(DATA_DIR_PATH, "rescaled_parsed_runs")
ENSEMBLE_PARSED_PROCESSED_DIR_PATH = os.path.join(PARSED_RUNS_PROCESSED_DIR_PATH, ensemble_id)
os.makedirs(PARSED_RUNS_PROCESSED_DIR_PATH, exist_ok=True)
os.makedirs(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, exist_ok=True)

In [11]:
# make sure file_names is sorted
files_names.sort()
files_names[0]

'1.csv'

In [14]:
time_period_ref = 8
baseline_id = 332332 #NOTE: Change this to your baseline id

# Build once (or use your all-ids file)
GLOBAL_ANCHOR = td.build_t0_anchor_from_file(
    path_csv=os.path.join(ENSEMBLE_PARSED_DIR_PATH, files_names[0]),  # or "all_ids.csv"
    time_period_ref=time_period_ref,
    baseline_id=baseline_id #NOTE: Change this to your baseline id
)

GLOBAL_ANCHOR

{'louisiana': emission_co2e_c2f6_ippu_product_use_product_use_ods_other              0.019325
 emission_co2e_c2f6_ippu_production_chemicals                           0.000000
 emission_co2e_c2f6_ippu_production_electronics                         0.000444
 emission_co2e_c2f6_ippu_production_metals                              0.000000
 emission_co2e_c2h3f3_ippu_product_use_product_use_ods_refrigeration    0.583135
                                                                          ...   
 emission_co2e_pfcs_ippu_production_other_product_manufacturing         0.000000
 emission_co2e_sf6_ippu_production_chemicals                            0.000000
 emission_co2e_sf6_ippu_production_electronics                          0.005307
 emission_co2e_sf6_ippu_production_metals                               0.000000
 emission_co2e_sf6_ippu_production_other_product_manufacturing          0.232438
 Name: louisiana_332332, Length: 568, dtype: float64}

In [15]:
for run, output_file in enumerate(files_names):
    path_in = os.path.join(ENSEMBLE_PARSED_DIR_PATH, output_file)
    df_in = pd.read_csv(path_in)

    # keep only years >= t0
    df_in = df_in[df_in["time_period"] >= time_period_ref].copy()
    if df_in.empty:
        print(f"[skip] {output_file} has no rows >= t0")
        continue

    region = df_in["region"].iloc[0]
    print(f"[start] run={run} file={output_file} region={region}")

    td.rescale(
        z=0,
        rall=np.array([region]),
        data_all=df_in,
        te_all=te_all,
        initial_conditions_id=[baseline_id],                 # keep this stable
        dir_output=ENSEMBLE_PARSED_PROCESSED_DIR_PATH,
        time_period_ref=time_period_ref,
        run=run,
        global_t0_anchor_by_region=GLOBAL_ANCHOR,       # <- the constant anchor
        global_baseline_id=baseline_id
    )


    print(f"[done] run={run} file={output_file} region={region}")


[start] run=0 file=1.csv region=louisiana
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-28t15;29;22.344855/louisiana_0.csv
[done] run=0 file=1.csv region=louisiana
[start] run=1 file=10.csv region=louisiana
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-28t15;29;22.344855/louisiana_1.csv
[done] run=1 file=10.csv region=louisiana
[start] run=2 file=100.csv region=louisiana
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-28t15;29;22.344855/louisiana_2.csv
[done] run=2 file=100.csv region=louisiana
[start] run=3 file=1000.csv region=louisiana
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-28t15;29;22.344855/louisiana_3.csv
[done] run=3 file=1000.csv region=louisi

In [16]:
# collect decomposed runs
files_out = [f for f in os.listdir(ENSEMBLE_PARSED_PROCESSED_DIR_PATH) if f.endswith(".csv")]
data_complete = pd.concat(
    [pd.read_csv(os.path.join(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, f)) for f in files_out],
    ignore_index=True
)

# (optional) dedupe exact duplicates
key_cols = ["region", "primary_id", "time_period"]
data_complete = data_complete.drop_duplicates(subset=key_cols + [c for c in data_complete.columns if c not in key_cols])

In [17]:
data_complete.head()

,Index,time_period,primary_id,region,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,...,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq
0,louisiana_402561,8,402561,louisiana,0.0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,...,13.094104,117.042622,45.130223,2.629950,3.138402,0.447204,-36.545088,2.050000,1.233181,0.0
1,louisiana_402561,9,402561,louisiana,0.0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,...,-45.480477,116.837704,46.193033,2.320538,3.190261,0.454418,-38.672990,2.128597,1.216383,0.0
2,louisiana_402561,10,402561,louisiana,0.0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,...,19.988759,115.314125,47.331158,2.068537,3.243628,0.461872,-40.414072,2.205906,1.187552,0.0
3,louisiana_402561,11,402561,louisiana,0.0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,...,20.513264,113.739702,48.529794,1.878471,3.297843,0.469516,-41.895018,2.281949,1.146455,0.0
4,louisiana_402561,12,402561,louisiana,0.0,349349.863281,64784.326959,76.171466,75188.085210,6374.554301,...,21.562334,112.153499,49.780739,1.752500,3.352747,0.477311,-43.198871,2.375646,1.098152,0.0


In [18]:
time_period_ref

8

In [19]:
data_complete.shape

(84112, 4011)

In [ ]:
# data_complete = td.recompute_subsector_totals(data_complete, te_all)

# # --- VALIDATE --- # TODO: This validation is not working properly yet
print(td.assert_equal_t0(data_complete, 8, mapped_only=False))           # base vars
print(td.assert_equal_t0_totals(data_complete, 8))    # subsector totals


True
True


In [21]:
# Filter some outlier runds
# primary_ids_to_remove = [354920, 355090, 354624]
primary_ids_to_remove = [403318, 403447, 403689]
data_complete = data_complete[~data_complete["primary_id"].isin(primary_ids_to_remove)]
data_complete.shape

(84028, 4011)

In [22]:
# check for fields full of nans
data_complete.isnull().sum()

Index                                 0
time_period                           0
primary_id                            0
region                                0
area_agrc_crops_bevs_and_spices       0
                                     ..
emission_co2e_subsector_total_trww    0
emission_co2e_subsector_total_frst    0
emission_co2e_subsector_total_lndu    0
emission_co2e_subsector_total_soil    0
emission_co2e_subsector_total_ccsq    0
Length: 4011, dtype: int64

In [23]:
data_complete.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84028 entries, 0 to 84111
Columns: 4011 entries, Index to emission_co2e_subsector_total_ccsq
dtypes: float64(4007), int64(2), object(2)
memory usage: 2.5+ GB


In [24]:
attr_primary_df = pd.read_csv(os.path.join(RUN_ENSEMBLE_DIR_PATH, "attribute_primary.csv"))
attr_primary_df.head()

,primary_id,design_id,strategy_id,future_id
0,332332,4,0,0
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3


In [25]:
df_merged = data_complete.merge(attr_primary_df[["primary_id", "strategy_id"]], on="primary_id", how="left")
df_merged.head()

,Index,time_period,primary_id,region,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,...,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,strategy_id
0,louisiana_402561,8,402561,louisiana,0.0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,...,117.042622,45.130223,2.629950,3.138402,0.447204,-36.545088,2.050000,1.233181,0.0,6004
1,louisiana_402561,9,402561,louisiana,0.0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,...,116.837704,46.193033,2.320538,3.190261,0.454418,-38.672990,2.128597,1.216383,0.0,6004
2,louisiana_402561,10,402561,louisiana,0.0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,...,115.314125,47.331158,2.068537,3.243628,0.461872,-40.414072,2.205906,1.187552,0.0,6004
3,louisiana_402561,11,402561,louisiana,0.0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,...,113.739702,48.529794,1.878471,3.297843,0.469516,-41.895018,2.281949,1.146455,0.0,6004
4,louisiana_402561,12,402561,louisiana,0.0,349349.863281,64784.326959,76.171466,75188.085210,6374.554301,...,112.153499,49.780739,1.752500,3.352747,0.477311,-43.198871,2.375646,1.098152,0.0,6004


In [26]:
# --- WRITE FINAL OUTPUT ---
final_name = f"sisepuede_results_IDE_{ensemble_id}.csv"
out_path = os.path.join(RUN_ENSEMBLE_DIR_PATH, final_name)
df_merged.to_csv(out_path, index=False)
print(f"[final] wrote {out_path}")

[final] wrote /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/ensemble_data/sisepuede_summary_results_run_sisepuede_run_2025-08-28t15;29;22.344855/sisepuede_results_IDE_2025-08-28t15;29;22.344855.csv
